# EndoScan ER — Phase 2b cloud run (Colab "Run all")

Runs in **Colab** (cloud disk + egress), **not** a laptop. It stages the real ER
data from the approved locators in `registry/data/sources.yaml`, runs the ER
pipeline (strict gate + human approval), and DVC-pushes artifacts to a
Google-Drive-backed local remote. A human reviews metrics/scorecard/cards before
the `endpoints.json` entry lands in its own PR. No regulatory-grade claims.

`CONFIRM` cells require the real source column names / CERAPP experimental file
before running.

In [ ]:
# 1) Environment: NumPy<2 so cmapPy works; clone repo; install.
!pip -q install 'numpy<2' cmapPy h5py pyarrow requests dvc 'scikit-learn>=1.8' pydantic pyyaml pandas
!git clone https://github.com/Rirkella/endoscan-platform.git
%cd endoscan-platform
!pip -q install -e packages/endoscan_core
import sys

sys.path.insert(0, "pipelines/endpoints/ER/staging")

In [ ]:
# 2) Mount Drive and configure a DVC LOCAL remote inside the synced folder.
from google.colab import drive

drive.mount("/content/drive")
!dvc remote add --local er_gdrive /content/drive/MyDrive/endoscan-dvc || true
!dvc remote default --local er_gdrive
# (Path lives in git-ignored .dvc/config.local; never committed.)

In [ ]:
# 3) Fetch approved sources (LINCS metadata + Level-5 MODZ gctx, CERAPP
#    experimental sets, PubChem mapping). URLs come from sources.yaml locators.
import gzip
import shutil
from pathlib import Path

import fetch

from endoscan_core.datasets import load_sources

loc = {l.name: l.url for s in load_sources().sources for l in s.locators}
raw = Path("data/raw")
raw.mkdir(parents=True, exist_ok=True)
for key in ["gse92742_sig_info", "gse92742_gene_info"]:
    fetch.fetch_url(loc[key], raw / (key + ".txt.gz"))
# The 19.9 GB MODZ gctx (Colab scratch; deleted after slicing):
fetch.fetch_url(loc["gse92742_level5_modz_gctx"], raw / "level5_modz.gctx.gz")
with gzip.open(raw / "level5_modz.gctx.gz", "rb") as fi, open(raw / "level5_modz.gctx", "wb") as fo:
    shutil.copyfileobj(fi, fo)

In [ ]:
# 4) CONFIRM real columns, then stage. Build sig_meta for ER-labelled compounds
#    (MCF7/A549), landmark gene ids (pr_is_lm==1) from gene_info, and CERAPP
#    experimental + PubChem mapping rows. Then call the tested build_* helpers.

# sig_meta: columns compound_key, sig_id, cell_id, pert_dose, pert_time
#   (join sig_info to your ER label compounds via PubChem mapping; CONFIRM cols)
# landmark_gene_ids: gene_info[gene_info.pr_is_lm==1].pr_gene_id  (978)
# feature_names: stable landmark gene symbols in the same order
# TRY-FIRST: stage_er can call fetch.clue_io_landmark_signatures(...) instead of
#   the gctx slice; on failure fall back to the gctx (the backbone).
out = Path("data/staged/er")
out.mkdir(parents=True, exist_ok=True)
# stage_er.build_lincs_parquet(sig_meta, raw/'level5_modz.gctx', landmark_gene_ids, feature_names, out/'lincs.parquet')
# stage_er.build_cerapp_csv(cerapp_experimental_rows, activity_col='<CONFIRM>', out/'cerapp.csv')
# stage_er.build_pubchem_csv(pubchem_rows, out/'pubchem.csv')
print("Staging: fill the CONFIRM seams above with the real columns, then run.")

In [ ]:
# 5) Run the ER pipeline on the staged data (strict gate; approve AFTER review).
import importlib.util

spec = importlib.util.spec_from_file_location("er_run", "pipelines/endpoints/ER/run.py")
er_run = importlib.util.module_from_spec(spec)
sys.modules["er_run"] = er_run
spec.loader.exec_module(er_run)
from endoscan_core.datasets import load_sources

cfg = er_run.load_config(Path("pipelines/endpoints/ER/config.real.example.yaml"))
# Review the dataset card + gate first with approved=False, then re-run approved=True.
res = er_run.run_pipeline(
    cfg,
    allow_list=load_sources(),
    data_dir=Path("data/staged/er"),
    thresholds=er_run.load_thresholds(Path("registry/data/quality_gates.yaml")),
    output_root=Path("."),
)
print(res.model_dump_json(indent=2))

In [ ]:
# 6) Review, then DVC-push binaries + stage the git text for a Phase-2b PR.
!cat models/ER/model_selection.md models/ER/metrics.json
!dvc add data/staged/er models/ER/model.pkl && dvc push
print(
    "Review metrics/scorecard/leakage/cards, then commit the .dvc pointers +\n"
    "metrics.json, feature_schema.json, cards, model_selection.*, endpoints.json\n"
    "in a separate PR. Do NOT overclaim; no regulatory-grade validation."
)